# Parkinson Voice Measurements - Exploratory Data Analysis (EDA)
Dataset: https://archive.ics.uci.edu/dataset/174/parkinsons

Basic EDA using **NumPy** and **Pandas** only. No machine learning, no diagnosis and no medical conclusions.
`status` is used only as a group label (0 = healthy group, 1 = Parkinson's group).

#1. Import Libraries and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the dataset directly from the UCI repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data'
data = pd.read_csv(url)

*If the link above does not work in Colab, download `parkinsons.data` from the UCI page, upload it
using the Files panel on the left, and use `data = pd.read_csv('parkinsons.data')` instead.*

#2. Creating a Dataset

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df.head()

In [ ]:
df.tail()

#3. Observing the Dataframe

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

#4. Data Cleaning Checks

###1. Checking for Null Values

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum().sum()   # total missing values in the whole dataset

###2. Checking for Duplicate Values

In [ ]:
df.duplicated().sum()

In [ ]:
df['name'].duplicated().sum()   # recording names that repeat

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.shape

#5. Answering the Questions

###Q1. How many records are in each status group?

In [ ]:
df['status'].value_counts()

In [ ]:
df['status'].value_counts(normalize = True) * 100   # percentage of each group

###Q2. Mean, Median, Minimum and Maximum of selected voice measurements

In [ ]:
selected = ['MDVP:Fo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Shimmer']

Using NumPy arrays for the calculations:

In [ ]:
for col in selected:
    arr = df[col].to_numpy()          # convert the column to a NumPy array
    print(col)
    print('  Mean   :', np.mean(arr))
    print('  Median :', np.median(arr))
    print('  Min    :', np.min(arr))
    print('  Max    :', np.max(arr))
    print()

The same result using Pandas:

In [ ]:
df[selected].agg(['mean', 'median', 'min', 'max'])

###Q3. Compare the average of these measurements by status

In [ ]:
df.groupby('status')[selected].mean()

In [ ]:
df.groupby('status')[selected].median()   # median is less affected by extreme values

###Q4. 10 highest values of one selected measurement (MDVP:Jitter(%))

In [ ]:
top10 = df.sort_values('MDVP:Jitter(%)', ascending = False).head(10)
top10[['name', 'status', 'MDVP:Jitter(%)', 'MDVP:Fo(Hz)', 'MDVP:Shimmer']]

In [ ]:
top10['status'].value_counts()   # which status groups do the top 10 come from?

Filtering example: rows with jitter above the 90th percentile

In [ ]:
p90 = np.percentile(df['MDVP:Jitter(%)'].to_numpy(), 90)
high_jitter = df[df['MDVP:Jitter(%)'] > p90]
print('90th percentile:', p90)
print('Rows above it  :', len(high_jitter))

###Q5. Missing values and duplicates

In [ ]:
print('Missing values  :', df.isnull().sum().sum())
print('Duplicate rows  :', df.duplicated().sum())

#6. Data Visualisation

###1. Status-count chart

In [ ]:
counts = df['status'].value_counts().sort_index()

plt.bar(['0 (Healthy)', "1 (Parkinson's group)"], counts.values)

plt.xlabel('status')
plt.ylabel('Number of Records')
plt.title('Records in Each Status Group')

plt.show()

###2. Histogram

In [ ]:
xaxis = df['MDVP:Fo(Hz)']

plt.hist(xaxis, bins = 20)

plt.xlabel('MDVP:Fo(Hz)')
plt.ylabel('Frequency')
plt.title('Histogram of MDVP:Fo(Hz)')

plt.show()

###3. Histogram of MDVP:Fo(Hz) split by status (optional)

In [ ]:
plt.hist(df[df['status'] == 0]['MDVP:Fo(Hz)'], bins = 15, alpha = 0.6, label = 'status 0')
plt.hist(df[df['status'] == 1]['MDVP:Fo(Hz)'], bins = 15, alpha = 0.6, label = 'status 1')

plt.xlabel('MDVP:Fo(Hz)')
plt.ylabel('Frequency')
plt.title('MDVP:Fo(Hz) by status')
plt.legend()

plt.show()

#7. Data Observations

The three observations below are generated from the data, so the numbers always match the output above.

In [ ]:
n0 = (df['status'] == 0).sum()
n1 = (df['status'] == 1).sum()
means = df.groupby('status')[selected].mean()
skew = df['MDVP:Jitter(%)'].skew()

print(f"1. The groups are imbalanced: status 1 has {n1} records and status 0 has {n0} records, "
      f"so raw counts should not be compared directly.")
print(f"2. Mean MDVP:Fo(Hz) is {means.loc[1, 'MDVP:Fo(Hz)']:.1f} for status 1 and {means.loc[0, 'MDVP:Fo(Hz)']:.1f} for status 0; "
      f"mean jitter is {means.loc[1, 'MDVP:Jitter(%)']:.5f} vs {means.loc[0, 'MDVP:Jitter(%)']:.5f}; "
      f"mean shimmer is {means.loc[1, 'MDVP:Shimmer']:.4f} vs {means.loc[0, 'MDVP:Shimmer']:.4f}. These are descriptive group differences only.")
print(f"3. MDVP:Jitter(%) is right-skewed (skewness = {skew:.2f}): its mean is larger than its median "
      f"and a few very high values stand out. The dataset has {df.isnull().sum().sum()} missing values and {df.duplicated().sum()} duplicate rows.")

**Note:** This is a basic descriptive EDA of an educational dataset. The recordings come from a small number of people
(several recordings each), and no diagnostic or predictive conclusions are drawn.